# ML Tutor — Week 9: DL starts: neural network fundamentals
**Track:** DL & GenAI · **Lab:** Lab 9 — Train a first ADR neural network

**Objectives this week:**
1. Explain how a perceptron becomes an MLP and how backpropagation trains it. *(CO2)*
2. Build and train a first neural network in PyTorch/Keras. *(CO5)*
3. Compare a neural ADR-signal classifier against a classical ML baseline. *(CO2, CO5)*

> Anti-shortcut rule: hints live in ML Tutor, revealed one tier at a time. Work here, check hints there. You are graded on unaided competence.


## Before you start (~25 min)
**Goal for this session:** Arrive ready to move from trees and distances into layered learning. The main job this week is to understand why neural nets are just another function learner, but one with many adjustable layers and a different training loop. Reference delta: teach one small neural model in the core path; deeper backprop detail is optional.

**Warm up — answer these from memory before scrolling on** (retrieval beats re-reading):
1. From Week 8, what is the difference between a cluster and a supervised prediction target?
2. From Week 7, why must model families be compared under the same evaluation rules?
3. What did we learn about scaling and preprocessing when model behavior depended on the representation?


In [ ]:
# SETUP — run me first (Shift+Enter)
import numpy as np
import pandas as pd

# Dataset: Synthetic ADR signal table with fictional features such as age_band, med_class, prior_adr, polypharmacy_count, prior_ed_visits, renal_score, and adr_flag. Teaching-only data with no PHI.
# PLACEHOLDER: the dataset for this week arrives with the course repo under data/week-09/ .
# If a lab step expects a file, download it from the ml-tutor-labs repo's data/week-09/ folder first.

print("Setup OK — numpy", np.__version__, "· pandas", pd.__version__)


## Worked example — Perceptron to MLP
**Lane A · the general idea:** A perceptron takes inputs, multiplies them by weights, adds a bias, and makes a simple decision. An MLP adds hidden layers so the learner can represent more complex boundaries.

**Lane B · the same idea in pharmacy terms:** For ADR signal detection, the first network is not magic. It is just a layered classifier that may capture interactions between medication history, comorbidity, and utilization patterns better than a shallow model.

Below is a tiny, complete demo of this idea. Run it, read every line, change one thing, run it again.


In [ ]:
# WORKED EXAMPLE — a neuron is a weighted sum + threshold (an ADR alert rule)
def neuron(inputs, weights, bias):
    total = sum(x * w for x, w in zip(inputs, weights)) + bias
    return 1 if total > 0 else 0        # "fire" = raise the ADR flag

# inputs: [potassium_high, on_ace_inhibitor, renal_impairment]  — 0/1 each
weights, bias = [2.0, 1.5, 1.0], -3.0   # learned from data in real life; hand-set here

print(neuron([1, 1, 0], weights, bias))   # 2 + 1.5 - 3 = 0.5 > 0 → fires (1)
print(neuron([1, 0, 0], weights, bias))   # 2 - 3 = -1      ≤ 0 → silent (0)
# an MLP is just LAYERS of these, with the weights learned from data.


## 1. Prepare the data for a neural model
Separate features from label, encode the input table, and set up an honest train/test split.

**Checkpoint:** Checkpoint 1 verifies: X and y are separated correctly, the split is stratified, and the preprocessing step does not peek at the test set.


**Predict before you run:** read the starter code below WITHOUT running it. Write one sentence: what do you expect to see printed or returned once the TODOs are filled in correctly? This step is double-scaffolded because it is your first neural-network lab — predicting first catches misunderstandings before they compile.


In [ ]:
# Your prediction (as a comment):
#



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Synthetic ADR signal panel (no PHI) — inline teaching data.
adr_df = pd.DataFrame([
    {"age_band": 1, "med_class": "nsaid", "prior_adr": 0, "polypharmacy_count": 2, "prior_ed_visits": 0, "renal_score": 0.2, "adr_flag": 0},
    {"age_band": 4, "med_class": "anticoagulant", "prior_adr": 1, "polypharmacy_count": 7, "prior_ed_visits": 2, "renal_score": 0.8, "adr_flag": 1},
    {"age_band": 2, "med_class": "nsaid", "prior_adr": 0, "polypharmacy_count": 3, "prior_ed_visits": 0, "renal_score": 0.3, "adr_flag": 0},
    {"age_band": 3, "med_class": "opioid", "prior_adr": 1, "polypharmacy_count": 6, "prior_ed_visits": 1, "renal_score": 0.6, "adr_flag": 1},
    {"age_band": 1, "med_class": "antibiotic", "prior_adr": 0, "polypharmacy_count": 1, "prior_ed_visits": 0, "renal_score": 0.15, "adr_flag": 0},
    {"age_band": 4, "med_class": "anticoagulant", "prior_adr": 1, "polypharmacy_count": 8, "prior_ed_visits": 3, "renal_score": 0.9, "adr_flag": 1},
    {"age_band": 2, "med_class": "antibiotic", "prior_adr": 0, "polypharmacy_count": 2, "prior_ed_visits": 0, "renal_score": 0.25, "adr_flag": 0},
    {"age_band": 3, "med_class": "opioid", "prior_adr": 1, "polypharmacy_count": 5, "prior_ed_visits": 1, "renal_score": 0.55, "adr_flag": 1},
    {"age_band": 1, "med_class": "nsaid", "prior_adr": 0, "polypharmacy_count": 2, "prior_ed_visits": 0, "renal_score": 0.2, "adr_flag": 0},
    {"age_band": 4, "med_class": "anticoagulant", "prior_adr": 1, "polypharmacy_count": 7, "prior_ed_visits": 2, "renal_score": 0.85, "adr_flag": 1},
])

target = "adr_flag"
feature_cols = [
    "age_band",
    "med_class",
    "prior_adr",
    "polypharmacy_count",
    "prior_ed_visits",
    "renal_score",
]

X = adr_df[feature_cols]
y = adr_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=7,
)

categorical_cols = ["age_band", "med_class"]
numeric_cols = ["prior_adr", "polypharmacy_count", "prior_ed_visits", "renal_score"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
    ]
)

# TODO: fit preprocess on X_train ONLY, then transform both splits into
# these two exact names (checkpoint verifies these, not the pipeline object):
X_train_t = ...
X_test_t = ...


**Self-check 1:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 2. Fit a small multilayer perceptron
Train a tiny MLP on the prepared data and compare it with a simpler baseline.

**Checkpoint:** Checkpoint 2 verifies: both models are fitted on the training set and the student compares the neural model with a baseline using the same evaluation rules.


**Predict before you run:** read the starter code below WITHOUT running it. Write one sentence: what do you expect to see printed or returned once the TODOs are filled in correctly? This step is double-scaffolded because it is your first neural-network lab — predicting first catches misunderstandings before they compile.


In [ ]:
# Your prediction (as a comment):
#



In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

mlp = Pipeline([
    ("prep", preprocess),
    ("clf", MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=7)),
])

baseline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

# TODO: fit both models and compare test-set performance
...


**Self-check 2:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 3. Inspect the training behavior
Look at loss or iteration behavior and note whether the network trained stably or needs adjustment.

**Checkpoint:** Checkpoint 3 verifies: the student can inspect at least one training-behavior signal and describe whether the model converged cleanly.


**Predict before you run:** read the starter code below WITHOUT running it. Write one sentence: what do you expect to see printed or returned once the TODOs are filled in correctly? This step is double-scaffolded because it is your first neural-network lab — predicting first catches misunderstandings before they compile.


In [ ]:
# Your prediction (as a comment):
#



In [ ]:
# TODO: inspect the fitted MLP training behavior
# If available, print the loss curve or the number of iterations.
print(mlp.named_steps["clf"].loss_)
...


**Self-check 3:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 4. Write the compare-and-justify summary
Explain which model you would keep, which one you would not, and why the comparison matters for ADR prediction.

**Checkpoint:** Checkpoint 4 verifies: the student can summarize the neural-vs-baseline comparison in plain language and name one reason the neural model is or is not justified.


**Predict before you run:** read the starter code below WITHOUT running it. Write one sentence: what do you expect to see printed or returned once the TODOs are filled in correctly? This step is double-scaffolded because it is your first neural-network lab — predicting first catches misunderstandings before they compile.


In [ ]:
# Your prediction (as a comment):
#



In [ ]:
summary = """
...
"""

print(summary)


**Self-check 4:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## Reflection — make it stick (5 min, write before you leave)
1. **Teach it:** explain your Step 4 code to a colleague in exactly 3 sentences — what goes in, what happens, what comes out. Write the sentences below.
2. **What would break this?** A classmate insists: *“Neural networks are always better because they are deep.”* Using what you just built, describe one concrete case from this week's lab where acting on that belief produces a wrong result — and how you would catch it.


In [ ]:
# Your reflection (write as comments)
# 1) Three sentences:
#
# 2) What would break this, concretely:
#



## Optional challenge — Homework: Neural-net comparison note
Write a short note comparing the first neural network with the baseline classifier on the ADR signal task. State which model you would keep for the teaching case, what evidence supports that choice, and what remains uncertain.

**Deliverable:** A short comparison note with the chosen metric and one training-behavior observation.


In [ ]:
# HW / challenge workspace — build it here

